In [ ]:
# ==========================================
# NLGCL Kaggle 运行指南 (All-in-One)
# ==========================================
# 本 Notebook 用于在 Kaggle 环境中配置并运行 NLGCL 模型。
# 
# ## 前置条件
# 1. **GPU 环境**: 请确保 Notebook 设置中 Accelerator 选择 GPU (如 GPU P100 或 T4)。
# 2. **数据集**: 请上传本地 `dataset` 文件夹到 Kaggle Datasets (命名推荐 `nlgcl-dataset`)，并挂载到 Notebook 中。
#    - Kaggle 挂载路径通常为 `/kaggle/input/nlgcl-dataset`。
#
# ------------------------------------------
# STEP 1: 环境检测与代码准备
# ------------------------------------------
import os
import shutil

# [关键修正] 使用你的个人仓库地址 (确保代码修改能生效)
GITHUB_REPO_URL = "https://github.com/yangzeha/NLGCL.git" 
REPO_NAME = "NLGCL" # 仓库名

# 消除 git 加载日志的辅助函数
def run_silent(cmd):
    os.system(f"{cmd} > /dev/null 2>&1")

if os.path.exists('/kaggle/working'):
    # 在 Kaggle 环境下
    # print("Detected Kaggle environment.") # 禁用非必要输出
    
    # 始终回到工作根目录，防止路径混乱
    os.chdir('/kaggle/working')
    
    # [强制清理] 如果目录已存在，先删除，确保 clone 的是最新的正确仓库
    # (之前的运行可能 clone 了别人的旧仓库，导致本地修改无效)
    if os.path.exists(REPO_NAME):
        # print(f"Removing existing {REPO_NAME} directory to force fresh clone...")
        shutil.rmtree(REPO_NAME)
    
    # 克隆代码
    cmd = f"git clone {GITHUB_REPO_URL} {REPO_NAME}"
    # print(f"Executing: {cmd}")
    run_silent(cmd)
    
    # 进入代码目录
    if os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
        # print(f"Changed directory to {os.getcwd()}")
        
        # [Kaggle 环境补丁] 创建缺失的目录以防止 recbole_gnn/utils.py 报错
        # 原因: utils.py 会尝试 import sequential_recommender，如果目录不存在则抛出 ModuleNotFoundError
        for missing_module in ["sequential_recommender", "social_recommender"]:
            target_dir = os.path.join("recbole_gnn", "model", missing_module)
            os.makedirs(target_dir, exist_ok=True)
            with open(os.path.join(target_dir, "__init__.py"), "w") as f:
                f.write(f"# {missing_module} module patch for Kaggle\n")
        # print("Applied patches for missing model directories.")
else:
    print("Not in Kaggle environment, assuming local run.")
    # 如果在本地，不需要 clone，直接运行即可

# ------------------------------------------
# STEP 2: 安装依赖 (极速版 - 优化下载体积)
# ------------------------------------------
# 为了解决 Kaggle 预装 PyTorch 版本过新导致的 PyG 编译缓慢问题，
# 我们降级到 PyTorch 2.5.1 以利用官方二进制包 (Whl)。
# 这个过程需要下载约 800MB 数据，在 Kaggle 网络下通常需要 1-2 分钟。
# 这是目前最快且最稳定的方案。

print("System initializing... (Please wait 1-2 minutes)")

# 使用 %%capture 隐藏冗长的安装日志，只显示关键信息
# 注意：在 Notebook 代码单元格中使用 %%capture 需要放在最开头，
# 这里我们用 Python 上下文管理器来模拟，或者直接忍受日志。
# 为了直观，我们保留日志但精简安装列表。

# 1. [核心修复] 降级 NumPy < 2.0
# RecBole 和旧版代码不兼容 NumPy 2.0 (移除了 np.float_ 等属性)
# print("Downgrading NumPy to < 2.0 for compatibility...")
!pip install -q "numpy<2.0" > /dev/null 2>&1

# 2. 安装 RecBole (会自动处理依赖，忽略 colorama 警告即可)
# 使用 -q (quiet) 减少输出
!pip install -q --upgrade recbole > /dev/null 2>&1

# 3. 这里的策略是：只安装 torch 核心包，不安装 torchvision/torchaudio 以节省时间和流量
#    NLGCL 是图推荐模型，通常不需要视觉和音频库
# print("Switching to stable PyTorch 2.5.1 (Core only) for fast setup...")
!pip install -q torch==2.5.1+cu124 --extra-index-url https://download.pytorch.org/whl/cu124 > /dev/null 2>&1

# 4. 秒装 PyG 二进制包
# print("Installing PyG binaries...")
wheel_url = "https://data.pyg.org/whl/torch-2.5.1+cu124.html"
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url} > /dev/null 2>&1
!pip install -q torch_geometric > /dev/null 2>&1

# 5. 安装其他依赖
!pip install -q -r requirements.txt > /dev/null 2>&1

import torch
import numpy as np
# print(f"Final PyTorch Version: {torch.__version__}")
# print(f"Final NumPy Version: {np.__version__}")
# print(f"CUDA Available: {torch.cuda.is_available()}")
print("Environment setup complete.")

# ------------------------------------------
# STEP 3: 数据集准备 (自动寻找与链接)
# ------------------------------------------
# 自动寻找 Kaggle Input 中的 dataset 目录
def find_dataset_root(search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if 'yelp.inter' in files:
            if os.path.basename(root) == 'yelp':
                return os.path.dirname(root)
            return os.path.dirname(root)
    return None

if os.path.exists('/kaggle/working'):
    current_dataset_link = 'dataset'
    
    # 1. 寻找挂载的数据集
    found_path = find_dataset_root()
    if not found_path:
        # 尝试默认路径
        potential_paths = [
            "/kaggle/input/nlgcl-dataset",
            "/kaggle/input/nlgcl-dataset/dataset",
            "/kaggle/input/dataset"
        ]
        for p in potential_paths:
            if os.path.exists(os.path.join(p, 'yelp', 'yelp.inter')):
                found_path = p
                break
    
    if found_path:
        # print(f"Found dataset at: {found_path}")
        
        # 2. 清理旧的 dataset 目录/链接
        if os.path.exists(current_dataset_link):
            if os.path.islink(current_dataset_link):
                os.unlink(current_dataset_link)
            else:
                shutil.rmtree(current_dataset_link)
        
        # 3. 复制数据集
        # print("Copying dataset to working directory (required for write access)...")
        shutil.copytree(found_path, current_dataset_link)
        # print("Dataset copied successfully.")
        
    else:
        print("Error: Could not find 'yelp/yelp.inter' in /kaggle/input.")
        !find /kaggle/input -maxdepth 3
else:
    print("Local environment, skipping dataset setup.")

# !ls -l dataset/yelp/

# ------------------------------------------
# STEP 4: 强制写入过滤配置 (覆盖 yelp.yaml)
# ------------------------------------------
yelp_config_content = """
load_col:
  inter: [user_id, item_id, rating]
ITEM_ID_FIELD: item_id
RATING_FIELD: rating

# 核心过滤配置：过滤交互少于 15 次的用户和物品
user_inter_num_interval: "[15,inf)"
item_inter_num_interval: "[15,inf)"

val_interval:
  rating: "[3,inf)"

warm_up_step: 40

# ----------------
# 训练与日志配置
# ----------------
epochs: 300
train_batch_size: 4096
learner: adam
learning_rate: 0.001
eval_step: 1
stopping_step: 20

# 评估指标
metrics: ["Recall", "NDCG"]
topk: [10, 20, 50]
valid_metric: NDCG@10

# 日志输出控制
verbose: True
state: INFO
show_progress: False  # 禁止显示进度条，只保留日志输出
"""

os.makedirs('properties', exist_ok=True)
with open('properties/yelp.yaml', 'w') as f:
    f.write(yelp_config_content)
# print("Updated properties/yelp.yaml with filtering rules and removed progress bar.")

# ------------------------------------------
# STEP 5: 运行训练 (自定义输出过滤)
# ------------------------------------------
import subprocess
import re
import sys

if os.path.exists("saved"):
    import glob
    for pth in glob.glob("saved/*.pth"):
        try:
            os.remove(pth)
        except:
            pass

print("Starting training with custom output filtering...")

# 确保 recbole 安装
try:
    import recbole
except ImportError:
    pass

# 定义命令 (使用 -u 禁用缓冲，确保实时输出)
cmd = ["python", "-u", "main.py", "--dataset=yelp"]

# 状态变量
print_enabled = False
# 正则匹配损失行: "train_loss1: 247.8721, train_loss2: 0.0002, train_loss3: 89.7495"
loss_pattern = re.compile(r"train_loss1: ([\d\.]+), train_loss2: ([\d\.]+), train_loss3: ([\d\.]+)")

# 使用 subprocess 运行并实时处理输出
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        # 1. 过滤逻辑：只在检测到 "The number of users" 后开始输出
        if "The number of users" in line:
            print_enabled = True
        
        if print_enabled:
            # 过滤掉包含 GPU RAM 的行
            if "GPU RAM" in line:
                continue
                
            # 2. 计算并显示 Total Loss
            # 原始格式: ... train_loss1: x, train_loss2: y, train_loss3: z]
            match = loss_pattern.search(line)
            if match:
                l1, l2, l3 = map(float, match.groups())
                total = l1 + l2 + l3
                # 去除行尾换行符，在括号前插入 total_loss
                # line 通常以 "]" 或 "]\n" 结尾
                line = line.strip()
                if line.endswith(']'):
                    line = line[:-1] + f", total_loss: {total:.4f}]"
                else:
                    line = line + f" (Total Loss: {total:.4f})"
                print(line) # print 会自动加换行
            else:
                print(line, end='') # 原样输出，不做改动

if p.returncode != 0:
    print(f"\nTraining finished with return code {p.returncode}")
else:
    print("\nTraining completed successfully.")